# 🛵 Challenge 16: 실시간 배달 알림 시스템

**난이도:** ⭐⭐⭐  
**사전 요구사항:** 노트북 03, 10  
**내비게이션:** [← 이전(15)](./15_challenge_saga_order.ipynb) | [다음(17) →](./17_challenge_image_pipeline.ipynb)

---

## 📖 시나리오

### 배달의민족 배달 추적 시스템
고객이 주문한 음식의 배달 상태를 실시간으로 추적합니다.

### 4단계 상태 전이
1. **주문 접수** (0분) - 가게에서 주문 확인
2. **조리 중** (+5분) - 음식 조리 시작
3. **배달 중** (+15분) - 라이더가 픽업 후 출발
4. **배달 완료** (+30분) - 고객에게 전달 완료

### 기술 구현
- **TTL + DLX**: 각 단계별 지연 시간 구현
- **Topic Routing**: 상태별 라우팅
- **Redis Pub/Sub**: 실시간 알림 푸시

## 🏗️ 아키텍처


<div align="center"> <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 280" width="100%" height="auto"> <rect width="100%" height="100%" fill="#F8FAFC" rx="12" /> <text x="400" y="30" font-family="-apple-system, sans-serif" font-size="14" font-weight="bold" fill="#0F172A" text-anchor="middle">실시간 배달 알림 시스템 아키텍처</text> <g transform="translate(30, 115)"> <circle cx="25" cy="25" r="18" fill="#E2E8F0" stroke="#475569" stroke-width="1.5" /> <text x="25" y="29" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#475569" text-anchor="middle">시작</text> </g> <path d="M 100 140 L 145 140" stroke="#64748B" stroke-width="2" /><polygon points="139,136 145,140 139,144" fill="#64748B" /> <g transform="translate(150, 95)"> <rect x="0" y="0" width="150" height="90" rx="8" fill="#FEE2E2" stroke="#B91C1C" stroke-width="1.5" /> <text x="75" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#991B1B" text-anchor="middle">지연 큐 (Cooking)</text> <text x="75" y="44" font-family="sans-serif" font-size="9" fill="#7F1D1D" text-anchor="middle">TTL: 5초 (시뮬레이션)</text> <rect x="15" y="58" width="120" height="22" rx="4" fill="#FFFFFF" stroke="#FCA5A5" /> <text x="75" y="72" font-family="sans-serif" font-size="8" fill="#991B1B" text-anchor="middle">상태: COOKING_COMPLETE</text> </g> <path d="M 300 140 L 365 140" stroke="#B91C1C" stroke-width="2" /><polygon points="359,136 365,140 359,144" fill="#B91C1C" /> <text x="332" y="128" font-family="sans-serif" font-size="8" fill="#991B1B" text-anchor="middle">만료 (DLX)</text> <g transform="translate(370, 110)"> <circle cx="30" cy="30" r="24" fill="#38BDF8" stroke="#0284C7" stroke-width="1.5" /> <text x="30" y="34" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#FFFFFF" text-anchor="middle">DLX</text> </g> <path d="M 430 140 L 495 140" stroke="#0284C7" stroke-width="2" /><polygon points="489,136 495,140 489,144" fill="#0284C7" /> <g transform="translate(500, 95)"> <rect x="0" y="0" width="130" height="90" rx="8" fill="#DCFCE7" stroke="#15803D" stroke-width="2" /> <text x="65" y="24" font-family="-apple-system, sans-serif" font-size="11" font-weight="bold" fill="#166534" text-anchor="middle">Redis Pub/Sub</text> <text x="65" y="44" font-family="sans-serif" font-size="9" fill="#14532D" text-anchor="middle">channel: "notify"</text> <rect x="10" y="58" width="110" height="22" rx="4" fill="#FFFFFF" stroke="#BBF7D0" /> <text x="65" y="72" font-family="sans-serif" font-size="8" fill="#166534" text-anchor="middle">실시간 푸시 발행</text> </g> <path d="M 630 140 L 695 140" stroke="#15803D" stroke-width="2" /><polygon points="689,136 695,140 689,144" fill="#15803D" /> <g transform="translate(700, 105)"> <rect x="0" y="0" width="80" height="70" rx="6" fill="#F3E8FF" stroke="#7E22CE" stroke-width="1.5" /> <text x="40" y="26" font-family="-apple-system, sans-serif" font-size="10" font-weight="bold" fill="#6B21A8" text-anchor="middle">고객 앱</text> <text x="40" y="42" font-family="sans-serif" font-size="8" fill="#581C87" text-anchor="middle">실시간 알림</text> <text x="40" y="54" font-family="sans-serif" font-size="8" fill="#581C87" text-anchor="middle">수신 🔔</text> </g> </svg> </div>


## 🎯 적용 패턴

1. **Delayed Message Pattern** - TTL + Dead Letter Exchange
2. **Event-Driven Architecture** - 상태 변경을 이벤트로 처리
3. **Pub/Sub Pattern** - Redis Pub/Sub으로 실시간 푸시
4. **State Transition** - 상태 머신 기반 배달 플로우

In [ ]:
# 환경 설정
import httpx
import json
import asyncio
from datetime import datetime, timedelta
from typing import Dict, List

# API 기본 URL
BASE_URL = "http://localhost:8000"

# Mock 데이터 로드
with open('../data/mock/delivery_timeline.json', 'r', encoding='utf-8') as f:
    MOCK_DATA = json.load(f)

print("✅ 환경 설정 완료")
print(f"📦 배달 데이터: {len(MOCK_DATA['deliveries'])}건")
print(f"⏱️ 총 소요시간: 약 30분 (시뮬레이션에서는 축소)")

## 🔧 Step 1: TTL + DLX 지연 큐 설정

### RabbitMQ Delayed Message 구조
1. **지연 큐** - TTL이 설정된 임시 큐
2. **DLX (Dead Letter Exchange)** - TTL 만료 시 메시지가 이동할 Exchange
3. **실제 큐** - DLX를 통해 메시지가 도착하는 최종 큐

### 각 단계별 지연 시간
- 주문접수 → 조리중: 5분 (시뮬레이션: 5초)
- 조리중 → 배달중: 10분 (시뮬레이션: 10초)
- 배달중 → 배달완료: 15분 (시뮬레이션: 15초)

In [ ]:
# 지연 큐 설정 헬퍼 함수
async def setup_delayed_queues():
    """TTL + DLX 기반 지연 큐 설정"""
    async with httpx.AsyncClient() as client:
        print("🔧 지연 큐 설정 중...\n")
        
        # 1. Topic Exchange에 큐 바인딩 (상태 변경 이벤트용)
        await client.post(
            f"{BASE_URL}/rabbitmq/topic/bind",
            params={"queue_name": "delivery.status.changed", "binding_key": "status.#"}
        )
        print("✅ Topic 바인딩 완료: delivery.status.changed (pattern: status.#)")
        
        # 2. 각 상태별 지연 큐 생성
        delays = {
            "cooking": 5000,      # 5초 (실제: 5분)
            "delivering": 10000,  # 10초 (실제: 10분)
            "delivered": 15000    # 15초 (실제: 15분)
        }
        
        for status, delay_ms in delays.items():
            # 지연 큐 생성 (실제 구현에서는 RabbitMQ Management API 사용)
            # 여기서는 개념적 설명을 위한 placeholder
            print(f"⏱️  {status} 지연 큐: {delay_ms}ms TTL")
        
        print("\n✅ 모든 지연 큐 설정 완료")

await setup_delayed_queues()

# 상태 정의
STATUS_CONFIG = {
    "order_received": {
        "name": "주문접수",
        "icon": "📝",
        "next_status": "cooking",
        "delay_seconds": 5
    },
    "cooking": {
        "name": "조리중",
        "icon": "🍳",
        "next_status": "delivering",
        "delay_seconds": 10
    },
    "delivering": {
        "name": "배달중",
        "icon": "🛵",
        "next_status": "delivered",
        "delay_seconds": 15
    },
    "delivered": {
        "name": "배달완료",
        "icon": "✅",
        "next_status": None,
        "delay_seconds": 0
    }
}

print("\n📋 상태 전이 순서:")
for status, config in STATUS_CONFIG.items():
    next_info = f"→ {config['next_status']}" if config['next_status'] else "(종료)"
    print(f"   {config['icon']} {config['name']} {next_info} (+{config['delay_seconds']}초)")

## 📤 Step 2: 상태 전이 메시지 발행

각 배달 건에 대해 초기 상태를 발행합니다.

In [ ]:
async def publish_delivery_order(delivery: Dict):
    """배달 주문 초기 상태 발행"""
    async with httpx.AsyncClient() as client:
        message = {
            "delivery_id": delivery['delivery_id'],
            "order_id": delivery['order_id'],
            "customer_name": delivery['customer_name'],
            "restaurant": delivery['restaurant'],
            "status": "order_received",
            "timestamp": datetime.now().isoformat()
        }
        
        # Redis에 초기 상태 저장
        key = f"delivery:{delivery['delivery_id']}:status"
        await client.post(
            f"{BASE_URL}/redis/kv/set",
            json={"key": key, "value": "order_received"}
        )
        
        # RabbitMQ Topic Exchange에 발행
        await client.post(
            f"{BASE_URL}/rabbitmq/topic/publish",
            json={
                "routing_key": "status.order_received",
                "content": json.dumps(message),
                "metadata": {}
            }
        )
        
        # 타임라인 기록
        timeline_key = f"delivery:{delivery['delivery_id']}:timeline"
        timeline_entry = {
            "status": "order_received",
            "timestamp": datetime.now().isoformat(),
            "message": f"{delivery['restaurant']}에서 주문을 접수했습니다"
        }
        await client.post(
            f"{BASE_URL}/redis/list/push",
            json={"key": timeline_key, "value": json.dumps(timeline_entry)}
        )
        
        return message

# 모든 배달 주문 발행
print("📤 배달 주문 발행 중...\n")

for delivery in MOCK_DATA['deliveries'][:3]:  # 처음 3건만 테스트
    message = await publish_delivery_order(delivery)
    print(f"✅ {delivery['delivery_id']}: {delivery['restaurant']} → {delivery['customer_name']}")

print(f"\n✅ 총 {len(MOCK_DATA['deliveries'][:3])}건 발행 완료")

## 📥 Step 3: Consumer + Pub/Sub 실시간 푸시

상태 변경 메시지를 소비하고 다음 단계를 예약합니다.

In [ ]:
async def process_status_transition(delivery_id: str, current_status: str):
    """상태 전이 처리 + 다음 단계 예약"""
    async with httpx.AsyncClient() as client:
        config = STATUS_CONFIG[current_status]
        
        # 현재 상태를 Redis에 저장
        status_key = f"delivery:{delivery_id}:status"
        await client.post(
            f"{BASE_URL}/redis/kv/set",
            json={"key": status_key, "value": current_status}
        )
        
        # 타임라인 추가
        timeline_key = f"delivery:{delivery_id}:timeline"
        timeline_entry = {
            "status": current_status,
            "timestamp": datetime.now().isoformat(),
            "message": f"{config['icon']} {config['name']}"
        }
        await client.post(
            f"{BASE_URL}/redis/list/push",
            json={"key": timeline_key, "value": json.dumps(timeline_entry)}
        )
        
        # Redis Pub/Sub으로 실시간 알림
        notification = {
            "delivery_id": delivery_id,
            "status": current_status,
            "message": timeline_entry['message'],
            "timestamp": timeline_entry['timestamp']
        }
        await client.post(
            f"{BASE_URL}/redis/pubsub/publish",
            json={
                "content": json.dumps(notification),
                "metadata": {"channel": f"delivery.notify.{delivery_id}"}
            }
        )
        
        print(f"   {config['icon']} {delivery_id}: {config['name']}")
        
        # 다음 단계가 있으면 지연 메시지 발행
        if config['next_status']:
            next_config = STATUS_CONFIG[config['next_status']]
            print(f"      ⏱️  {next_config['delay_seconds']}초 후 → {next_config['name']}")
            
            # 비동기적으로 다음 단계 예약
            asyncio.create_task(
                schedule_next_status(delivery_id, config['next_status'], next_config['delay_seconds'])
            )

async def schedule_next_status(delivery_id: str, next_status: str, delay_seconds: int):
    """다음 상태 전이 예약"""
    await asyncio.sleep(delay_seconds)
    await process_status_transition(delivery_id, next_status)

# 시뮬레이션 시작
print("🎬 배달 상태 추적 시뮬레이션 시작\n")
print("=" * 60)

# 각 배달에 대해 상태 전이 시작
tasks = []
for delivery in MOCK_DATA['deliveries'][:3]:
    task = asyncio.create_task(
        process_status_transition(delivery['delivery_id'], "order_received")
    )
    tasks.append(task)

# 모든 초기 상태 처리 대기
await asyncio.gather(*tasks)

print("\n⏳ 상태 전이 진행 중... (최대 30초 소요)")
print("   실시간으로 상태가 업데이트됩니다.\n")

# 모든 배달이 완료될 때까지 대기 (30초)
await asyncio.sleep(35)

## 🔍 검증: 배달 타임라인 조회

각 배달의 전체 상태 변경 이력을 확인합니다.

In [ ]:
async def display_delivery_timeline(delivery_id: str):
    """배달 타임라인 조회 및 표시"""
    async with httpx.AsyncClient() as client:
        # 타임라인 조회
        timeline_key = f"delivery:{delivery_id}:timeline"
        response = await client.get(
            f"{BASE_URL}/redis/list/range",
            params={"key": timeline_key, "start": 0, "stop": -1}
        )
        
        if response.status_code != 200:
            print(f"❌ {delivery_id}: 타임라인 없음")
            return
        
        timeline = response.json()['values']
        
        print(f"\n📦 {delivery_id} 배달 타임라인")
        print("=" * 60)
        
        # 역순으로 표시 (최신이 위로)
        for entry_str in reversed(timeline):
            entry = json.loads(entry_str)
            timestamp = datetime.fromisoformat(entry['timestamp'])
            time_str = timestamp.strftime("%H:%M:%S")
            print(f"   {time_str} | {entry['message']}")

# 모든 배달 타임라인 조회
for delivery in MOCK_DATA['deliveries'][:3]:
    await display_delivery_timeline(delivery['delivery_id'])

print("\n" + "=" * 60)
print("✅ 모든 배달이 완료되었습니다!")

## 💡 핵심 포인트

### 1. Delayed Messaging
**RabbitMQ TTL + DLX 패턴:**
```
Message → [Queue with TTL] → (expire) → DLX → Target Queue
```

**장점:**
- 정확한 지연 시간 제어
- Cron보다 이벤트 기반으로 더 유연
- 메시지 손실 방지 (지속성)

### 2. Event-Driven Architecture
- 각 상태 변경이 독립적인 이벤트
- 서비스 간 느슨한 결합
- 새로운 subscriber 추가가 쉬움

### 3. Redis Pub/Sub
**vs RabbitMQ:**
| 항목 | Redis Pub/Sub | RabbitMQ |
|------|---------------|----------|
| 속도 | 매우 빠름 | 빠름 |
| 지속성 | 없음 | 있음 |
| 용도 | 실시간 알림 | 안정적 메시징 |

### 4. State Machine 패턴
```python
# 상태 전이 규칙
TRANSITIONS = {
    'order_received': ['cooking'],
    'cooking': ['delivering'],
    'delivering': ['delivered'],
    'delivered': []  # 최종 상태
}
```

## 🚀 확장 아이디어

### 1. GPS 실시간 추적
```python
# 라이더 위치를 1초마다 업데이트
async def track_rider_location(delivery_id: str):
    while status == 'delivering':
        location = get_gps_location()
        await redis.publish(
            f"delivery.location.{delivery_id}",
            json.dumps(location)
        )
        await asyncio.sleep(1)
```

### 2. ETA (도착 예정 시간) 예측
- 과거 배달 데이터 기반 ML 모델
- 실시간 교통 상황 반영
- 라이더 속도 패턴 분석

### 3. 푸시 알림 통합
```python
# FCM (Firebase Cloud Messaging)
await send_push_notification(
    user_id=customer_id,
    title="배달 중 🛵",
    body="라이더가 음식을 픽업했습니다. 곧 도착합니다!"
)
```

### 4. 배달 지연 자동 감지
```python
# 예상 시간 초과 시 알림
if actual_time > expected_time + threshold:
    await notify_delay(delivery_id, reason="교통 체증")
```

In [ ]:
# 정리: Redis 데이터 삭제
async def cleanup():
    async with httpx.AsyncClient() as client:
        for delivery in MOCK_DATA['deliveries'][:3]:
            delivery_id = delivery['delivery_id']
            
            # 상태 및 타임라인 삭제
            keys = [
                f"delivery:{delivery_id}:status",
                f"delivery:{delivery_id}:timeline"
            ]
            
            for key in keys:
                await client.delete(
                    f"{BASE_URL}/redis/kv/delete/{key}"
                )

await cleanup()
print("🧹 정리 완료")

## 🧭 내비게이션

[← 이전 (15)](./15_challenge_saga_order.ipynb) | [다음 (17) →](./17_challenge_image_pipeline.ipynb)

---

**학습 완료!** 🎉  
다음 노트북에서는 Python + Go 하이브리드 이미지 처리 파이프라인을 구축합니다.